# On-Policy DPO on `with_refusals` SFT

Trains on-policy DPO on top of the `with_refusals` SFT adapter (instead of `baseline`).
All four phases run in a single notebook; only one model is on GPU at a time.

| Phase | Task | Time |
|---|---|---|
| 1 | Train Safety Reward Model | ~15 min |
| 2 | Generate responses with `with_refusals` student | ~30–60 min |
| 3 | Score responses with frozen RM | ~15 min |
| 4 | On-policy DPO training | ~30–60 min |

**Before running:**
1. Enable GPU: Notebook settings → Accelerator → **T4 x2** or **P100**
2. Enable Internet: Notebook settings → Internet → **On**
3. Add HuggingFace token: Add-ons → Secrets → **HF_TOKEN** (toggle On)

## Step 0: Install Dependencies

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets

## Step 1: Setup

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import gc, json, csv, io, shutil, torch, urllib.request
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import PeftModel, LoraConfig, get_peft_model, TaskType

BASE_MODEL_ID    = "meta-llama/Llama-3.2-1B"
REPO_URL         = "https://github.com/202520030411/Distill-Safety-Aligned-Models.git"
SFT_ADAPTER_DIR  = "repo/outputs/with_refusals"
DPO_PAIRS_PATH   = "repo/data/dpo_pairs.jsonl"

RM_OUTPUT_DIR                = "/kaggle/working/outputs/rm_with_refusals"
SCORED_OUTPUT_PATH           = "/kaggle/working/advbench_scored_with_refusals.jsonl"
ON_POLICY_DPO_OUTPUT_DIR     = "/kaggle/working/outputs/on_policy_dpo_with_refusals"
os.makedirs(RM_OUTPUT_DIR, exist_ok=True)
os.makedirs(ON_POLICY_DPO_OUTPUT_DIR, exist_ok=True)

REFUSAL_PREFIXES = [
    "I cannot", "I can't", "I'm not able", "I won't", "I'm unable",
    "I don't", "I apologize", "As an AI", "I'm sorry", "I must decline",
    "I'm not going to", "That's not something", "I am not able",
]

def is_refusal(text):
    t = text.lower()
    return any(p.lower() in t for p in REFUSAL_PREFIXES)

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
print(f"SFT adapter: {SFT_ADAPTER_DIR}")

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN loaded")

In [ ]:
!git clone {REPO_URL} repo
!ls repo/

## Step 2: Load Data

In [ ]:
pairs = load_jsonl(DPO_PAIRS_PATH)
print(f"Loaded {len(pairs)} preference pairs")

def format_pair(ex):
    prompt = ex["prompt"]
    return {
        "chosen":   f"Human: {prompt}\n\nAssistant: {ex['chosen']}",
        "rejected": f"Human: {prompt}\n\nAssistant: {ex['rejected']}",
    }

rm_ds = Dataset.from_list([format_pair(p) for p in pairs])
split = rm_ds.train_test_split(test_size=0.1, seed=42)
rm_train_ds = split["train"]
rm_eval_ds  = split["test"]
print(f"RM train: {len(rm_train_ds)}  eval: {len(rm_eval_ds)}")

In [ ]:
ADVBENCH_URL = (
    "https://raw.githubusercontent.com/llm-attacks/llm-attacks/main"
    "/data/advbench/harmful_behaviors.csv"
)
with urllib.request.urlopen(ADVBENCH_URL) as resp:
    reader = csv.DictReader(io.StringIO(resp.read().decode()))
    advbench_prompts = [row["goal"] for row in reader]

print(f"AdvBench prompts: {len(advbench_prompts)}")
print("Example:", advbench_prompts[0])

## Phase 1 — Train Safety Reward Model

RM training does not depend on the SFT adapter; it learns from the preference pairs directly.

⏱ ~15 min

In [ ]:
print("=" * 60)
print("PHASE 1: Training Safety Reward Model")
print("=" * 60)

rm_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=os.environ["HF_TOKEN"])
rm_tokenizer.pad_token = rm_tokenizer.eos_token
rm_tokenizer.padding_side = "right"

rm_base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_ID,
    num_labels=1,
    torch_dtype=torch.float32,
    device_map={"":0},
    token=os.environ["HF_TOKEN"],
)
rm_base.config.pad_token_id = rm_tokenizer.pad_token_id

rm_lora = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    modules_to_save=["score"],
    lora_dropout=0.05, bias="none", task_type=TaskType.SEQ_CLS,
)
rm_model = get_peft_model(rm_base, rm_lora)
rm_model.print_trainable_parameters()

In [ ]:
NUM_EPOCHS = 2
GRAD_ACCUM = 8
LR = 5e-5
MAX_LEN = 256
MAX_GRAD_NORM = 1.0

optimizer = torch.optim.AdamW(
    [p for p in rm_model.parameters() if p.requires_grad], lr=LR
)

rm_model.train()
global_step = 0
accum_loss = 0.0

print(f"Starting manual RM training: {len(rm_train_ds)} pairs, "
      f"{NUM_EPOCHS} epochs, grad_accum={GRAD_ACCUM}")

for epoch in range(NUM_EPOCHS):
    indices = torch.randperm(len(rm_train_ds)).tolist()
    for i, idx in enumerate(indices):
        row = rm_train_ds[idx]

        chosen_tok = rm_tokenizer(
            row["chosen"], return_tensors="pt",
            truncation=True, max_length=MAX_LEN, padding="max_length",
        ).to(rm_model.device)
        rejected_tok = rm_tokenizer(
            row["rejected"], return_tensors="pt",
            truncation=True, max_length=MAX_LEN, padding="max_length",
        ).to(rm_model.device)

        r_chosen  = rm_model(**chosen_tok).logits[0, 0].float()
        r_rejected = rm_model(**rejected_tok).logits[0, 0].float()
        loss = -F.logsigmoid(r_chosen - r_rejected) / GRAD_ACCUM

        loss.backward()
        accum_loss += loss.item()

        if (i + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(rm_model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1
            print(f"  epoch {epoch+1}/{NUM_EPOCHS}  step {global_step}  "
                  f"loss={accum_loss:.4f}  r_chosen={r_chosen.item():.3f}  "
                  f"r_rejected={r_rejected.item():.3f}")
            accum_loss = 0.0

    optimizer.step()
    optimizer.zero_grad()
    print(f"Epoch {epoch+1} done.")

rm_model.save_pretrained(RM_OUTPUT_DIR)
rm_tokenizer.save_pretrained(RM_OUTPUT_DIR)
print(f"RM saved to {RM_OUTPUT_DIR}")

del optimizer, rm_model, rm_base
free_gpu()
print("GPU freed after RM training")

## Phase 2 — Generate Responses with `with_refusals` Student

Generates 4 completions per AdvBench prompt using the `with_refusals` SFT student.

⏱ ~30–60 min

In [ ]:
print("=" * 60)
print("PHASE 2: Generating responses to AdvBench prompts")
print("=" * 60)

gen_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=os.environ["HF_TOKEN"])
gen_tokenizer.pad_token = gen_tokenizer.eos_token
gen_tokenizer.padding_side = "left"

gen_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"":0},
    token=os.environ["HF_TOKEN"],
)
gen_model = PeftModel.from_pretrained(gen_base, SFT_ADAPTER_DIR)
gen_model = gen_model.merge_and_unload()
gen_model.eval()
print(f"SFT adapter ({SFT_ADAPTER_DIR}) loaded and merged")

In [ ]:
NUM_GENERATIONS = 4
PROMPT_TEMPLATE = "Human: {}\n\nAssistant:"
responses_path = "/kaggle/working/advbench_responses_with_refusals.jsonl"

with open(responses_path, "w") as fout:
    for i, prompt in enumerate(advbench_prompts):
        full_prompt = PROMPT_TEMPLATE.format(prompt)
        inputs = gen_tokenizer(
            full_prompt, return_tensors="pt", truncation=True, max_length=256,
        ).to(gen_model.device)

        with torch.no_grad():
            outputs = gen_model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                num_return_sequences=NUM_GENERATIONS,
                pad_token_id=gen_tokenizer.pad_token_id,
            )

        responses = []
        for seq in outputs:
            text = gen_tokenizer.decode(seq[inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            responses.append(text.strip())

        fout.write(json.dumps({"prompt": prompt, "responses": responses}) + "\n")

        if (i + 1) % 50 == 0:
            print(f"  Generated {i+1}/{len(advbench_prompts)} prompts")

print(f"Done. Saved {len(advbench_prompts)} x {NUM_GENERATIONS} responses to {responses_path}")

del gen_model, gen_base
free_gpu()
print("GPU freed after generation")

## Phase 3 — Score Responses with Frozen RM

⏱ ~15 min

In [ ]:
print("=" * 60)
print("PHASE 3: Scoring responses with frozen RM")
print("=" * 60)

score_base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_ID,
    num_labels=1,
    torch_dtype=torch.float16,
    device_map={"":0},
    token=os.environ["HF_TOKEN"],
)
score_base.config.pad_token_id = rm_tokenizer.pad_token_id
score_model = PeftModel.from_pretrained(score_base, RM_OUTPUT_DIR)
score_model.eval()
for p in score_model.parameters():
    p.requires_grad = False
print("Frozen RM loaded for scoring")

In [ ]:
generated = load_jsonl(responses_path)
print(f"Scoring {len(generated)} prompts x {NUM_GENERATIONS} responses...")

with open(SCORED_OUTPUT_PATH, "w") as fout:
    for i, row in enumerate(generated):
        prompt = row["prompt"]
        rewards = []
        for resp in row["responses"]:
            full_text = f"Human: {prompt}\n\nAssistant: {resp}"
            inputs = rm_tokenizer(
                full_text, return_tensors="pt",
                truncation=True, max_length=256, padding=True,
            ).to(score_model.device)

            with torch.no_grad():
                rm_score = score_model(**inputs).logits[0, 0].item()

            heuristic = 0.5 if is_refusal(resp) else -0.5
            rewards.append(round(rm_score + heuristic, 4))

        fout.write(json.dumps({
            "prompt": prompt,
            "responses": row["responses"],
            "rewards": rewards,
        }) + "\n")

        if (i + 1) % 100 == 0:
            print(f"  Scored {i+1}/{len(generated)}")

print(f"Done. Saved scored data to {SCORED_OUTPUT_PATH}")

del score_model, score_base
free_gpu()

In [ ]:
sample = load_jsonl(SCORED_OUTPUT_PATH)[:3]
for s in sample:
    print(f"Prompt: {s['prompt'][:60]}...")
    for r, rw in zip(s['responses'], s['rewards']):
        tag = "REFUSE" if is_refusal(r) else "COMPLY"
        print(f"  [{tag}] reward={rw:.3f}  {r[:60]}...")
    print()

## Phase 4 — Build Preference Pairs + On-Policy DPO Training

Converts RM-scored responses into best-vs-worst preference pairs, then trains DPO
on top of `with_refusals`.

⏱ ~30–60 min

In [ ]:
PROMPT_TEMPLATE = "Human: {}\n\nAssistant:"
scored_data = load_jsonl(SCORED_OUTPUT_PATH)

preference_pairs = []
for row in scored_data:
    rewards = row["rewards"]
    responses = row["responses"]

    best_idx  = max(range(len(rewards)), key=lambda i: rewards[i])
    worst_idx = min(range(len(rewards)), key=lambda i: rewards[i])

    if rewards[best_idx] <= rewards[worst_idx]:
        continue

    preference_pairs.append({
        "prompt":   PROMPT_TEMPLATE.format(row["prompt"]),
        "chosen":   responses[best_idx],
        "rejected": responses[worst_idx],
    })

print(f"Valid preference pairs: {len(preference_pairs)}")
print(f"Skipped (no reward difference): {len(scored_data) - len(preference_pairs)}")

if preference_pairs:
    ex = preference_pairs[0]
    print(f"\nExample:")
    print(f"  Prompt:   {ex['prompt'][:60]}...")
    print(f"  Chosen:   {ex['chosen'][:60]}...")
    print(f"  Rejected: {ex['rejected'][:60]}...")

In [ ]:
print("Merging with_refusals SFT into base weights...")
MERGED_DIR = "/kaggle/working/merged_sft_with_refusals"

merge_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float32,
    device_map={"":0},
    attn_implementation="eager",
    token=os.environ["HF_TOKEN"],
)
merge_base = PeftModel.from_pretrained(merge_base, SFT_ADAPTER_DIR)
merge_base = merge_base.merge_and_unload()
merge_base.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}")

del merge_base
free_gpu()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=os.environ["HF_TOKEN"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    torch_dtype=torch.float32,
    device_map={"":0},
    attn_implementation="eager",
)

dpo_lora = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, dpo_lora)
model.print_trainable_parameters()

In [ ]:
def get_logprobs(model, input_ids, attention_mask, labels):
    """Compute per-token log probs for the response portion (where labels != -100)."""
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :]
    target = labels[:, 1:]
    mask = (target != -100).float()
    safe_target = target.clamp(min=0)
    log_probs = F.log_softmax(logits, dim=-1)
    token_logprobs = log_probs.gather(2, safe_target.unsqueeze(2)).squeeze(2)
    return (token_logprobs * mask).sum(dim=1)


def tokenize_pair(tokenizer, prompt, response, max_len=256):
    """Tokenize prompt+response, with labels=-100 for the prompt portion."""
    full_text = prompt + " " + response
    encoded = tokenizer(
        full_text, return_tensors="pt",
        truncation=True, max_length=max_len, padding="max_length",
    )
    prompt_enc = tokenizer(prompt, add_special_tokens=False)
    prompt_len = min(len(prompt_enc["input_ids"]), max_len)

    labels = encoded["input_ids"].clone()
    labels[0, :prompt_len] = -100
    pad_mask = encoded["attention_mask"][0] == 0
    labels[0, pad_mask] = -100

    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": labels,
    }

print("Helpers defined.")

In [ ]:
NUM_EPOCHS = 2
GRAD_ACCUM = 8
LR = 5e-5
BETA = 0.1
MAX_LEN = 256
MAX_GRAD_NORM = 1.0

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR
)

model.train()
global_step = 0
accum_loss = 0.0

print(f"Starting DPO training: {len(preference_pairs)} pairs, "
      f"{NUM_EPOCHS} epochs, grad_accum={GRAD_ACCUM}, beta={BETA}")
print(f"Optimizer steps per epoch: ~{len(preference_pairs) // GRAD_ACCUM}")

for epoch in range(NUM_EPOCHS):
    indices = torch.randperm(len(preference_pairs)).tolist()
    for i, idx in enumerate(indices):
        pair = preference_pairs[idx]
        prompt = pair["prompt"]

        chosen_tok   = tokenize_pair(tokenizer, prompt, pair["chosen"], MAX_LEN)
        rejected_tok = tokenize_pair(tokenizer, prompt, pair["rejected"], MAX_LEN)

        device = model.device
        c_ids  = chosen_tok["input_ids"].to(device)
        c_mask = chosen_tok["attention_mask"].to(device)
        c_lbl  = chosen_tok["labels"].to(device)
        r_ids  = rejected_tok["input_ids"].to(device)
        r_mask = rejected_tok["attention_mask"].to(device)
        r_lbl  = rejected_tok["labels"].to(device)

        pi_chosen  = get_logprobs(model, c_ids, c_mask, c_lbl)
        pi_rejected = get_logprobs(model, r_ids, r_mask, r_lbl)

        with torch.no_grad():
            model.disable_adapter_layers()
            ref_chosen  = get_logprobs(model, c_ids, c_mask, c_lbl)
            ref_rejected = get_logprobs(model, r_ids, r_mask, r_lbl)
            model.enable_adapter_layers()

        logits_diff = BETA * (
            (pi_chosen - ref_chosen) - (pi_rejected - ref_rejected)
        )
        loss = -F.logsigmoid(logits_diff).mean() / GRAD_ACCUM

        loss.backward()
        accum_loss += loss.item()

        if (i + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1
            print(f"  epoch {epoch+1}/{NUM_EPOCHS}  step {global_step}  "
                  f"loss={accum_loss:.4f}  "
                  f"reward_margin={logits_diff.item()/BETA:.3f}")
            accum_loss = 0.0

    optimizer.step()
    optimizer.zero_grad()
    print(f"Epoch {epoch+1} done.")

model.save_pretrained(ON_POLICY_DPO_OUTPUT_DIR)
tokenizer.save_pretrained(ON_POLICY_DPO_OUTPUT_DIR)
print(f"Adapter saved to {ON_POLICY_DPO_OUTPUT_DIR}")

## Step 5: Download Outputs

In [ ]:
shutil.make_archive("/kaggle/working/rm_with_refusals_adapter", "zip", RM_OUTPUT_DIR)
shutil.make_archive("/kaggle/working/on_policy_dpo_with_refusals", "zip", ON_POLICY_DPO_OUTPUT_DIR)
print("Download from the Output tab:")
print("  rm_with_refusals_adapter.zip")
print("  on_policy_dpo_with_refusals.zip")
print(f"  {SCORED_OUTPUT_PATH} (already in /kaggle/working/)")